# Truy xuat thong tin bang BM25

In [35]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math
import numpy as np


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## BM25 stem

In [19]:
def preprocess(tok, stemmer=ps, punctlist=puncts, stopwords=stoplist):
  tok = tok.lower()
  if tok.isdigit():
    return None
  if tok.isnumeric():
    return None
  if tok in punctlist:
    return None
  if tok in stopwords:
    return None
  return stemmer.stem(tok)

In [20]:
def indexing(src, idx="ind"):
  if src[-1] != '/':
    src += '/'
  # schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
  schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=KeywordAnalyzer()))
  ix = create_in(idx, schema)
  writer = ix.writer()

  files = os.listdir(src)
  for f in files:
    r = open(src + f, encoding="cp1252")
    terms = []
    for s in r:
      for sent in sent_tokenize(s.strip()):
        for tok in word_tokenize(sent):
          tok = preprocess(tok)
          if tok != None:
            terms.append(tok)
    r.close()
    cont = " ".join(terms)
    writer.add_document(docid="{}".format(f.split(".")[0]), content=cont)
  writer.commit()

In [21]:
indexing("Cranfield/Cranfield", "ind")

In [3]:
def readGroundTruth(src):
  if src[-1] != '/':
    src += '/'

  GT = {}
  for f in os.listdir(src):
    r = open(src + f)
    rel = {}
    for s in r:
      s = s.strip()
      sp = s.split("\t")
      if len(sp) < 2:
        continue
      did = sp[0].split(" ")[1]
      rel[did] = int(sp[1])
    GT[f.split(".")[0]] = rel
    r.close()
  return GT

In [22]:
GroundTruth = readGroundTruth("Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [23]:
def readQuery(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [24]:
Queries = readQuery("Cranfield/query.txt")
print(Queries)

{'1': 'similar law must obey construct aeroelast model heat high speed aircraft', '2': 'structur aeroelast problem associ flight high speed aircraft', '3': 'problem heat conduct composit slab solv far', '4': 'criterion develop show empir valid flow solut chemic react ga mixtur base simplifi assumpt instantan local chemic equilibrium', '5': 'chemic kinet system applic hyperson aerodynam problem', '6': 'theoret experiment guid turbul couett flow behaviour', '7': 'possibl relat avail pressur distribut ogiv forebodi zero angl attack lower surfac pressur equival ogiv forebodi angl attack', '8': 'method -dash exact approxim -dash present avail predict bodi pressur angl attack', '9': 'paper intern /slip flow/ heat transfer studi', '10': 'real-ga transport properti air avail wide rang enthalpi densiti', '11': 'possibl find analyt similar solut strong blast wave problem newtonian approxim', '12': 'aerodynam perform channel flow ground effect machin calcul', '13': 'basic mechan transon aileron b

In [25]:
def processQueries(ind, qry):

  idx = index.open_dir(ind)
  searcher = idx.searcher(weighting=scoring.BM25F())
  parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

  RET = {}
  for key in qry:
    query = parser.parse(qry[key])
    results = searcher.search(query, limit=None)
    rel = {}
    for i in range(len(results)):
      rel[results[i]["docid"]] = results[i].score
    RET[key] = rel
  return RET

In [26]:
RunResults = processQueries("ind", Queries)

In [27]:
import pytrec_eval

# GroundTruth: dict {qid: {docid: relevance}}
# RunResults: dict {qid: {docid: score}}

# Chọn các metric cần tính
evaluator = pytrec_eval.RelevanceEvaluator(
    GroundTruth,
    {
        "map",       # Mean Average Precision
        "infAP",     # Inferred MAP
        "11pt_avg",
        "P_5", "P_10", "P_20",  # Precision@k
        "recall_5", "recall_10", "recall_20"  # Recall@k
    }
)

# Tính kết quả
results = evaluator.evaluate(RunResults)

# In kết quả từng query
for qid, metrics in results.items():
    print(f"Query {qid}")
    for m, v in metrics.items():
        print(f"  {m:10s}: {v:.4f}")
    print("-"*30)


Query 1
  map       : 0.2190
  P_5       : 0.6000
  P_10      : 0.3000
  P_20      : 0.3500
  recall_5  : 0.1071
  recall_10 : 0.1071
  recall_20 : 0.2500
  infAP     : 0.2545
  11pt_avg  : 0.2659
------------------------------
Query 2
  map       : 0.2148
  P_5       : 0.6000
  P_10      : 0.4000
  P_20      : 0.3000
  recall_5  : 0.1250
  recall_10 : 0.1667
  recall_20 : 0.2500
  infAP     : 0.2184
  11pt_avg  : 0.2476
------------------------------
Query 3
  map       : 0.4890
  P_5       : 0.6000
  P_10      : 0.5000
  P_20      : 0.2500
  recall_5  : 0.5000
  recall_10 : 0.8333
  recall_20 : 0.8333
  infAP     : 0.6467
  11pt_avg  : 0.6062
------------------------------
Query 4
  map       : 0.5476
  P_5       : 0.2000
  P_10      : 0.1000
  P_20      : 0.0500
  recall_5  : 0.5000
  recall_10 : 0.5000
  recall_20 : 0.5000
  infAP     : 0.5714
  11pt_avg  : 0.5887
------------------------------
Query 5
  map       : 0.3283
  P_5       : 0.4000
  P_10      : 0.3000
  P_20      : 0.1

In [28]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")

Average over all queries:
  map       : 0.2765
  P_5       : 0.2738
  P_10      : 0.2138
  P_20      : 0.1504
  recall_5  : 0.2477
  recall_10 : 0.3603
  recall_20 : 0.4842
  infAP     : 0.3239
  11pt_avg  : 0.2983
  F1_5      : 0.2601
  F1_10     : 0.2683
  F1_20     : 0.2296


## BM25 Lemma

In [5]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [6]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN


In [7]:
from nltk import pos_tag

def preprocess_lemma(tok, punctlist=puncts, stopwords=stoplist):
    tok = tok.lower()

    if tok.isdigit() or tok.isnumeric():
        return None
    if tok in punctlist:
        return None
    if tok in stopwords:
        return None

    pos = pos_tag([tok])[0][1]
    wn_pos = get_wordnet_pos(pos)

    return lemmatizer.lemmatize(tok, wn_pos)

In [18]:
import os
import shutil
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT, STORED
from whoosh.analysis import KeywordAnalyzer

def indexing_lemma(src, idx="ind"):
    # 🔴 XÓA INDEX CŨ
    if os.path.exists(idx):
        shutil.rmtree(idx)

    os.mkdir(idx)

    if src[-1] != '/':
        src += '/'

    schema = Schema(
        docid=STORED(),
        content=TEXT(stored=True, analyzer=StandardAnalyzer())
    )

    ix = create_in(idx, schema)
    writer = ix.writer()

    files = os.listdir(src)
    for f in files:
        with open(src + f, encoding="cp1252") as r:
            terms = []
            for s in r:
                for sent in sent_tokenize(s.strip()):
                    for tok in word_tokenize(sent):
                        tok = preprocess(tok)
                        if tok is not None:
                            terms.append(tok)

        writer.add_document(
            docid=f.split(".")[0],
            content=" ".join(terms)
        )

    writer.commit()


In [9]:
indexing_lemma("Cranfield/Cranfield", "ind")

In [10]:
GroundTruth = readGroundTruth("Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [11]:
def readQuery_lemma(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess_lemma(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [12]:
Queries = readQuery_lemma("Cranfield/query.txt")
print(Queries)

{'1': 'similarity law must obeyed construct aeroelastic model heat high speed aircraft', '2': 'structural aeroelastic problem associate flight high speed aircraft', '3': 'problem heat conduction composite slab solve far', '4': 'criterion developed show empirically validity flow solution chemically react gas mixture base simplify assumption instantaneous local chemical equilibrium', '5': 'chemical kinetic system applicable hypersonic aerodynamic problem', '6': 'theoretical experimental guide turbulent couette flow behaviour', '7': 'possible relate available pressure distribution ogive forebody zero angle attack low surface pressure equivalent ogive forebody angle attack', '8': 'method -dash exact approximate -dash presently available predict body pressure angle attack', '9': 'paper internal /slip flow/ heat transfer study', '10': 'real-gas transport property air available wide range enthalpy density', '11': 'possible find analytical similar solution strong blast wave problem newtonian a

In [15]:
RunResults = processQueries("ind", Queries)

In [16]:
import pytrec_eval

# GroundTruth: dict {qid: {docid: relevance}}
# RunResults: dict {qid: {docid: score}}

# Chọn các metric cần tính
evaluator = pytrec_eval.RelevanceEvaluator(
    GroundTruth,
    {
        "map",       # Mean Average Precision
        "infAP",     # Inferred MAP
        "11pt_avg",
        "P_5", "P_10", "P_20",  # Precision@k
        "recall_5", "recall_10", "recall_20"  # Recall@k
    }
)

# Tính kết quả
results = evaluator.evaluate(RunResults)

# In kết quả từng query
for qid, metrics in results.items():
    print(f"Query {qid}")
    for m, v in metrics.items():
        print(f"  {m:10s}: {v:.4f}")
    print("-"*30)


Query 1
  map       : 0.1911
  P_5       : 0.4000
  P_10      : 0.4000
  P_20      : 0.3000
  recall_5  : 0.0714
  recall_10 : 0.1429
  recall_20 : 0.2143
  infAP     : 0.2222
  11pt_avg  : 0.2278
------------------------------
Query 2
  map       : 0.1113
  P_5       : 0.4000
  P_10      : 0.2000
  P_20      : 0.2000
  recall_5  : 0.0833
  recall_10 : 0.0833
  recall_20 : 0.1667
  infAP     : 0.1130
  11pt_avg  : 0.1528
------------------------------
Query 3
  map       : 0.3633
  P_5       : 0.2000
  P_10      : 0.3000
  P_20      : 0.2500
  recall_5  : 0.1667
  recall_10 : 0.5000
  recall_20 : 0.8333
  infAP     : 0.4250
  11pt_avg  : 0.3836
------------------------------
Query 4
  map       : 0.1979
  P_5       : 0.2000
  P_10      : 0.1000
  P_20      : 0.0500
  recall_5  : 0.5000
  recall_10 : 0.5000
  recall_20 : 0.5000
  infAP     : 0.2135
  11pt_avg  : 0.2102
------------------------------
Query 5
  map       : 0.0025
  P_5       : 0.0000
  P_10      : 0.0000
  P_20      : 0.0

In [17]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")

Average over all queries:
  map       : 0.1498
  P_5       : 0.1600
  P_10      : 0.1307
  P_20      : 0.0987
  recall_5  : 0.1271
  recall_10 : 0.2049
  recall_20 : 0.2950
  infAP     : 0.1735
  11pt_avg  : nan
  F1_5      : 0.1416
  F1_10     : 0.1596
  F1_20     : 0.1479


## BM25 Lay thong tin phan hoi

In [111]:
def preprocess(tok, stemmer=ps, punctlist=puncts, stopwords=stoplist):
    tok = tok.lower()
    if tok.isdigit():
        return None
    if tok.isnumeric():
        return None
    if tok in punctlist:
        return None
    if tok in stopwords:
        return None
    return stemmer.stem(tok)

In [114]:
def indexing(src, idx="ind"):
  if src[-1] != '/':
    src += '/'
  schema = Schema(docid=ID(stored=True, unique=True), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
  ix = create_in(idx, schema)
  writer = ix.writer()

  files = os.listdir(src)
  for f in files:
    r = open(src + f, encoding="cp1252")
    terms = []
    for s in r:
      for sent in sent_tokenize(s.strip()):
        for tok in word_tokenize(sent):
          tok = preprocess(tok)
          if tok != None:
            terms.append(tok)
    r.close()
    cont = " ".join(terms)
    writer.add_document(docid="{}".format(f.split(".")[0]), content=cont)
  writer.commit()

In [115]:
indexing("Cranfield/Cranfield", "ind")

In [116]:
GroundTruth = readGroundTruth("Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [202]:
Queries = readQuery("Cranfield/query.txt")
print(Queries)

{'1': 'similar law must obey construct aeroelast model heat high speed aircraft', '2': 'structur aeroelast problem associ flight high speed aircraft', '3': 'problem heat conduct composit slab solv far', '4': 'criterion develop show empir valid flow solut chemic react ga mixtur base simplifi assumpt instantan local chemic equilibrium', '5': 'chemic kinet system applic hyperson aerodynam problem', '6': 'theoret experiment guid turbul couett flow behaviour', '7': 'possibl relat avail pressur distribut ogiv forebodi zero angl attack lower surfac pressur equival ogiv forebodi angl attack', '8': 'method -dash exact approxim -dash present avail predict bodi pressur angl attack', '9': 'paper intern /slip flow/ heat transfer studi', '10': 'real-ga transport properti air avail wide rang enthalpi densiti', '11': 'possibl find analyt similar solut strong blast wave problem newtonian approxim', '12': 'aerodynam perform channel flow ground effect machin calcul', '13': 'basic mechan transon aileron b

In [119]:
def preprocess_query(q):
    toks = []
    for sent in sent_tokenize(q):
        for tok in word_tokenize(sent):
            tok = preprocess(tok)
            if tok:
                toks.append(tok)
    return " ".join(toks)


In [120]:
def processQueries(ind, qry):
    idx = index.open_dir(ind)
    searcher = idx.searcher(weighting=scoring.BM25F())
    parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

    RET = {}
    for key in qry:
        query = parser.parse(qry[key])
        results = searcher.search(query, limit=None)
        rel = {}
        for i in range(len(results)):
            rel[results[i]["docid"]] = results[i].score
            RET[key] = rel
    return RET

In [121]:
qid = list(Queries.keys())[0]
print("RAW :", Queries[qid])
print("PROC:", preprocess_query(Queries[qid]))

ret = processQueries("ind", {qid: Queries[qid]})
print("Retrieved docs:", len(ret[qid]))

RAW : similar law must obey construct aeroelast model heat high speed aircraft
PROC: similar law must obey construct aeroelast model heat high speed aircraft
Retrieved docs: 834


In [161]:
from whoosh import index, scoring, qparser

def bm25_retrieve(ind, queries, k1=1.2, b=0.75):
    idx = index.open_dir(ind)
    searcher = idx.searcher(weighting=scoring.BM25F(k1=k1, B=b))
    parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

    runs = {}
    for qid, qtext in queries.items():
        q = parser.parse(qtext)
        results = searcher.search(q, limit=None)
        # Lưu docnum để truy xuất nội dung cực nhanh trong bước PRF
        runs[qid] = [hit.docnum for hit in results] 

    return runs


In [ ]:
def get_pseudo_relevant(runs, K=10):
    RD = {}
    for qid in runs:
        RD[qid] = runs[qid][:K]
    return RD

In [163]:
from collections import Counter

def collect_term_stats(searcher, RD):
    term_df = Counter()
    R = 0

    for qid in RD:
        for docnum in RD[qid]:
            # Truy xuất trực tiếp bằng docnum nội bộ
            doc = searcher.stored_fields(docnum)
            
            if "content" not in doc:
                continue

            terms = set(doc["content"].split())
            for t in terms:
                term_df[t] += 1
            R += 1

    return term_df, R

In [203]:
def estimate_p(term_df, R, K_smooth=0.75, p_prior=0.5):
    p = {}
    for term, df in term_df.items():
        p[term] = (df + K_smooth * p_prior) / (R + K_smooth)
    return p


In [204]:
def compute_idf(searcher, term):
    N = searcher.doc_count()
    df = searcher.doc_frequency("content", term)
    return np.log((N - df + 0.5) / (df + 0.5))


In [205]:
import math

def compute_weights(searcher, p_terms):
    weights = {}

    for term, p in p_terms.items():
        if 0 < p < 1:
            idf = compute_idf(searcher, term)
            weights[term] = idf + math.log(p / (1 - p))

    return weights


In [ ]:
def expand_query_safe(original_query, weights, top_m=5, expansion_weight_scale=0.5):
    """
    original_query: Chuỗi văn bản thô (chưa có boost)
    weights: Thống kê trọng số từ PRF
    expansion_weight_scale: Hệ số điều chỉnh độ tin cậy của từ mới (0.1 - 0.5)
    """
    original_terms = original_query.lower().split()
    # Boost từ gốc cố định để giữ đúng ý định ban đầu (Original Intent)
    q_terms = [f"{t}^2.0" for t in original_terms]

    if not weights:
        return " ".join(q_terms)

    # Lọc bỏ các từ đã có trong query gốc trước khi lấy top_m
    filtered_weights = {t: w for t, w in weights.items() if t not in original_terms}
    
    if not filtered_weights:
        return " ".join(q_terms)

    max_w = max(filtered_weights.values())
    sorted_terms = sorted(filtered_weights.items(), key=lambda x: x[1], reverse=True)[:top_m]

    for term, w in sorted_terms:
        if w <= 0: continue
        
        # Boost cho từ mới = (tỉ lệ so với max) * hệ số tin cậy
        # Điều này đảm bảo từ mới không bao giờ quan trọng bằng từ gốc
        boost = (w / max_w) * expansion_weight_scale
        q_terms.append(f"{term}^{round(boost, 2)}")

    return " ".join(q_terms)

In [199]:
len(Queries)

225

In [238]:
from collections import defaultdict

def bm25_prf_iterative_with_history(
    ind,
    queries,
    k1=1.2,
    b=0.4,
    K=10,
    top_m=5,
    max_iter=2,  # Khuyến nghị 1 hoặc 2
    eps=1e-4
):
    idx = index.open_dir(ind)
    # Dùng BM25F cho searcher
    searcher = idx.searcher(weighting=scoring.BM25F(k1=k1, B=b))
    parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

    # raw_queries: Luôn giữ văn bản gốc sạch sẽ
    raw_queries = queries.copy() 
    # current_expanded_queries: Dùng để truy vấn lấy tài liệu phản hồi
    current_expanded_queries = queries.copy()

    p_history = defaultdict(list)
    w_history = defaultdict(list)

    for it in range(max_iter):
        print(f"--- Iteration {it+1}/{max_iter} ---")

        # 1. Retrieval để lấy tập phản hồi giả định (Pseudo-relevant Docs)
        runs = bm25_retrieve(ind, current_expanded_queries, k1, b)
        RD = get_pseudo_relevant(runs, K)

        new_expanded_queries = {}

        for qid in raw_queries:
            # Lấy văn bản gốc của query này
            original_text = raw_queries[qid]
            
            # 2. Thu thập thống kê từ tập RD
            term_df, R = collect_term_stats(searcher, {qid: RD[qid]})

            if R == 0:
                new_expanded_queries[qid] = current_expanded_queries[qid]
                continue

            # 3. Ước lượng p(t|R) và tính trọng số
            p_curr = estimate_p(term_df, R)
            weights = compute_weights(searcher, p_curr)
            
            p_history[qid].append(p_curr)
            w_history[qid].append(weights)

            # 4. Mở rộng từ văn bản GỐC (Tránh lỗi term^2.0^2.0)
            new_expanded_queries[qid] = expand_query_safe(
                original_text, 
                weights, 
                top_m=top_m
            )

        # Cập nhật query mở rộng cho vòng lặp tiếp theo
        current_expanded_queries = new_expanded_queries

    # --- Final Retrieval ---
    print("Executing final retrieval...")
    final_run = {}
    for qid, qtext in current_expanded_queries.items():
        q = parser.parse(qtext)
        results = searcher.search(q, limit=None) # Thường lấy top 1000 cho evaluation
        final_run[qid] = {str(r["docid"]): r.score for r in results}

    return final_run, p_history, w_history

In [226]:
from matplotlib import pyplot as plt


def plot_p_history(p_history, qid, top_terms=5):
    """
    p_history: dict[qid] -> list of p_dicts
    """
    history = p_history[qid]

    if not history:
        print("No history to plot")
        return

    last_p = history[-1]
    terms = sorted(last_p, key=last_p.get, reverse=True)[:top_terms]

    for t in terms:
        values = [p.get(t, 0) for p in history]
        plt.plot(range(len(history)), values, marker="o", label=t)

    plt.xlabel("Iteration")
    plt.ylabel("p(t | R)")
    plt.title(f"Evolution of p(t) for query {qid}")
    plt.legend()
    plt.grid(True)
    plt.show()


In [227]:
def plot_w_history(w_history, qid, top_terms=5):
    """
    w_history: dict[qid] -> list of weight_dicts
    """
    history = w_history[qid]

    if not history:
        print("No history to plot")
        return

    last_w = history[-1]
    terms = sorted(last_w, key=last_w.get, reverse=True)[:top_terms]

    for t in terms:
        values = [w.get(t, 0) for w in history]
        plt.plot(range(len(history)), values, marker="o", label=t)

    plt.xlabel("Iteration")
    plt.ylabel("w(t)")
    plt.title(f"Evolution of w(t) for query {qid}")
    plt.legend()
    plt.grid(True)
    plt.show()


In [215]:
with index.open_dir("ind").searcher() as s:
    print(list(s.all_stored_fields())[:5])

[{'content': 'experiment investig aerodynam wing slipstream experiment studi wing propel slipstream made order determin spanwis distribut lift increas due slipstream differ angl attack wing differ free stream slipstream veloc ratio result intend part evalu basi differ theoret treatment problem compar span load curv togeth support evid show substanti part lift increment produc slipstream due /destalling/ boundari layer control effect integr remain lift increment subtract destal lift found agre well potenti flow theori empir evalu destal effect made specif configur experi', 'docid': '1'}, {'content': 'theori impact tube low pressur theoret analysi made impact tube relat free stream mach number impact free stream pressur densiti extrem low pressur shown result differ appreci correspond continuum relat', 'docid': '10'}, {'content': 'vibrat isol aircraft power plant vibrat aircraft structur almost alway trace vibratori forc origin power plant forc transmit aircraft two way .. ( ) action air

In [253]:

# 2. Run PRF on tuning queries
final_run, p_hist, w_hist = bm25_prf_iterative_with_history(
    "ind",
    Queries,
    k1=1.2,
    b=0.75,
    K=20,
    top_m=5,
    max_iter=2,
    eps=1e-4
)

--- Iteration 1/2 ---
--- Iteration 2/2 ---
Executing final retrieval...


In [ ]:
import math
import pytrec_eval

def evaluate_run(run_results, ground_truth):
    """
    run_results: {qid: {docid: score, ...}, ...}
    ground_truth: {qid: {docid: rel_level, ...}, ...}
    """
    metrics = [
        "map", "infAP", "11pt_avg",
        "P_5", "P_10", "P_20",
        "recall_5", "recall_10", "recall_20"
    ]
    
    # Khởi tạo evaluator
    evaluator = pytrec_eval.RelevanceEvaluator(ground_truth, metrics)
    results = evaluator.evaluate(run_results)

    # Khởi tạo các biến tích lũy
    summary = {m: 0.0 for m in metrics}
    valid_queries = 0

    for qid, res in results.items():
        if not math.isnan(res["map"]):
            valid_queries += 1
            for m in metrics:
                summary[m] += res[m]

    if valid_queries == 0:
        return None

    # Tính trung bình
    for m in metrics:
        summary[m] /= valid_queries
    
    # Tính thêm F1-score
    def calc_f1(p, r):
        return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)
    
    summary["F1_5"] = calc_f1(summary["P_5"], summary["recall_5"])
    summary["F1_10"] = calc_f1(summary["P_10"], summary["recall_10"])
    summary["F1_20"] = calc_f1(summary["P_20"], summary["recall_20"])
    summary["count"] = valid_queries

    return summary

def print_metrics(summary, title="Evaluation Results"):
    if summary is None:
        print("No valid queries to evaluate.")
        return

    print(f"=== {title} ===")
    print(f"Queries evaluated : {summary['count']}")
    print(f"MAP        : {summary['map']:.4f}")
    print(f"P@10       : {summary['P_10']:.4f}")
    print(f"infAP      : {summary['infAP']:.4f}")
    print(f"11pt Avg   : {summary['11pt_avg']:.4f}")
    print("-" * 32)

    for k in [5, 10, 20]:
        print(
            f"P@{k:<2} : {summary[f'P_{k}']:.4f} | "
            f"R@{k:<2} : {summary[f'recall_{k}']:.4f} | "
            f"F1@{k:<2} : {summary[f'F1_{k}']:.4f}"
        )
    print()

In [254]:
# 2. Lọc GroundTruth tương ứng với tập Tune
GT_tune = {qid: GroundTruth[qid] for qid in Queries if qid in GroundTruth}
print(GT_tune)
# 3. Đánh giá
summary_prf = evaluate_run(final_run, GT_tune)

if summary_prf:
    print_metrics(summary_prf, title="BM25 + PRF (Tuning Set)")

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '2': {'12': 1, '15': 2, '184': 2, '858': 2, '51': 3, '102': 3, '202': 3, '14': 4, '52': 4, '380': 4, '746': 1, '859': 2, '948': 2, '285': 3, '390': 3, '391': 3, '442': 4, '497': 3, '643': 3, '856': 3, '857': 3, '877': 3, '864': 3, '658': 3, '486': -1}, '3': {'90': 3, '91': 3, '119': 3, '144': 3, '181': 3, '399': 3, '485': -1}, '4': {'236': 3, '166': 3, '488': -1}, '5': {'552': 1, '401': 3, '1297': 3, '1296': 1, '488': -1}, '6': {'99': 2, '115': 3, '257': 3, '258': 3, '491': -1}, '7': {'20': 2, '56': 3, '57': 3, '58': 3, '19': 4, '492': -1}, '8': {'48': 1, '122': 1, '20': 3, '58': 3, '196': 3, '354': 1, '360': 1, '197': 3, '999': 3, '1112': 3, '1005': 1, '492': -1}, '9': {'21': 2, '22': 2, '550': 2, '534': 